In [2]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
from collections import Counter

from gensim.models import Word2Vec, FastText

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
from pathlib import Path

ROOT = Path.cwd()

if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "cleaned_tinystories.csv"
MODEL_DIR = ROOT / "models"

print("Project Root:", ROOT)
print("Dataset Path:", DATA_PATH)
print("Model Directory:", MODEL_DIR)

Project Root: c:\Users\hp\Desktop\nlp-project-tinyGpt
Dataset Path: c:\Users\hp\Desktop\nlp-project-tinyGpt\data\cleaned_tinystories.csv
Model Directory: c:\Users\hp\Desktop\nlp-project-tinyGpt\models


In [6]:

result = subprocess.run(
    [sys.executable, str(ROOT / "data" / "download_tinystories.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dataset download failed.")

print("Raw dataset downloaded successfully!")


scanned 20,000 | Dialogue=5000 MoralValue=646 Conflict=597
scanned 60,000 | Dialogue=5000 MoralValue=1936 Conflict=1656
scanned 140,000 | Dialogue=5000 MoralValue=4643 Conflict=3911

Scanned 176,551 stories. Tag frequencies:
  <none>          77,462
  Dialogue        70,206
  Twist           20,008
  MoralValue      13,619
  BadEnding       11,835
  Foreshadowing   11,821
  Conflict        11,740

Saved 15,000 stories -> C:\Users\hp\Desktop\nlp-project-tinyGpt\data\raw_tinystories.csv
narrative_type
Conflict      5000
Dialogue      5000
MoralValue    5000

Raw dataset downloaded successfully!


In [ ]:

result = subprocess.run(
    [sys.executable, str(ROOT / "src" / "clean_pipeline.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Data cleaning failed.")

print("Dataset cleaning completed!")


In [ ]:

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
print("\nAvailable Columns:")
print(df.columns.tolist())

df.head()


In [5]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, str(ROOT / "data" / "download_tinystories.py")],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

scanned 20,000 | Dialogue=5000 MoralValue=646 Conflict=597
scanned 60,000 | Dialogue=5000 MoralValue=1936 Conflict=1656
scanned 140,000 | Dialogue=5000 MoralValue=4643 Conflict=3911

Scanned 176,551 stories. Tag frequencies:
  <none>          77,462
  Dialogue        70,206
  Twist           20,008
  MoralValue      13,619
  BadEnding       11,835
  Foreshadowing   11,821
  Conflict        11,740

Saved 15,000 stories -> C:\Users\hp\Desktop\nlp-project-tinyGpt\data\raw_tinystories.csv
narrative_type
Conflict      5000
Dialogue      5000
MoralValue    5000



In [ ]:

sentences = [
    text.split()
    for text in df["text_processed"].fillna("")
    if text.strip()
]

total_tokens = sum(len(sentence) for sentence in sentences)

print("Total Stories:", len(sentences))
print("Total Tokens:", f"{total_tokens:,}")
print("First Sentence:", sentences[0][:20])


In [ ]:

PARAMS = {
    "vector_size": 100,
    "window": 5,
    "min_count": 5,
    "epochs": 10,
    "workers": 4,
    "seed": 42
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Hyperparameters:")
print(PARAMS)


In [ ]:

start_time = time.time()

cbow = Word2Vec(
    sentences=sentences,
    sg=0,
    **PARAMS
)

cbow_time = time.time() - start_time

cbow.save(str(MODEL_DIR / "cbow.model"))

print("CBOW Training Completed!")
print("Vocabulary Size:", len(cbow.wv))
print("Training Time:", round(cbow_time, 2), "seconds")


In [ ]:

start_time = time.time()

skipgram = Word2Vec(
    sentences=sentences,
    sg=1,
    **PARAMS
)

skipgram_time = time.time() - start_time

skipgram.save(str(MODEL_DIR / "skipgram.model"))

print("Skip-gram Training Completed!")
print("Vocabulary Size:", len(skipgram.wv))
print("Training Time:", round(skipgram_time, 2), "seconds")


In [ ]:

comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Architecture": "Context to Target",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Architecture": "Target to Context",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    }
])

comparison


In [ ]:

test_words = ["king", "queen", "story", "happy", "boy"]

for word in test_words:
    print(f"\nWord: {word}")

    if word in cbow.wv:
        print("CBOW:", cbow.wv.most_similar(word, topn=5))
    else:
        print("CBOW: Word not in vocabulary")

    if word in skipgram.wv:
        print("Skip-gram:", skipgram.wv.most_similar(word, topn=5))
    else:
        print("Skip-gram: Word not in vocabulary")


In [ ]:

word_counts = Counter(
    word
    for sentence in sentences
    for word in sentence
)

rare_words = [
    word for word, count in word_counts.items()
    if count == 2 and len(word) > 6
][:10]

print("Rare Words:")
print(rare_words)


In [ ]:

start_time = time.time()

ft = FastText(
    sentences=sentences,
    sg=1,
    min_n=3,
    max_n=6,
    bucket=200_000,
    **PARAMS
)

fasttext_time = time.time() - start_time

ft.save(str(MODEL_DIR / "fasttext.model"))

print("fastText Training Completed!")
print("Vocabulary Size:", len(ft.wv))
print("Training Time:", round(fasttext_time, 2), "seconds")


In [ ]:

test_words = ["pianist", "dinosaurr", "happpy", "puppys"]

for word in test_words:
    print(f"\nTesting Word: {word}")

    if word in cbow.wv:
        print("Word2Vec:", cbow.wv.most_similar(word, topn=3))
    else:
        print("Word2Vec: No vector available")

    print(
        "fastText:",
        ft.wv.most_similar(word, topn=3)
    )


In [ ]:

final_comparison = pd.DataFrame([
    {
        "Model": "CBOW",
        "Vocabulary": len(cbow.wv),
        "Training Time": round(cbow_time, 2)
    },
    {
        "Model": "Skip-gram",
        "Vocabulary": len(skipgram.wv),
        "Training Time": round(skipgram_time, 2)
    },
    {
        "Model": "fastText",
        "Vocabulary": len(ft.wv),
        "Training Time": round(fasttext_time, 2)
    }
])

final_comparison


In [ ]:

print("Saved Model Files:")

for file in MODEL_DIR.glob("*.model"):
    print(file.name, "-", round(file.stat().st_size / (1024 * 1024), 2), "MB")


In [ ]:

# Conclusion

In this assignment, three word embedding models were trained using the TinyStories dataset.

1. **CBOW:** Predicts a target word using its surrounding context. It is generally faster to train.
2. **Skip-gram:** Predicts surrounding context words using a target word. It can perform better on less frequent words.
3. **fastText:** Uses character-level n-grams to generate word representations. It can produce vectors for words that are not present in the training vocabulary.

All three models use 100-dimensional word vectors, a context window of 5, minimum word frequency of 5, and 10 training epochs.

The models were saved for future use and comparison.
